In [1]:
# Cell 1: Environment Setup & GPU Verification (TDD)
# Project: HallucinationLegalRAGChatbots
#
# Clean Code: Thin orchestration. All logic in src/environment.py.
# No side effects on import — checks wrapped in run_environment_checks().
#
# Reproducibility: src/repro.configure() MUST be called first — before any
# import of torch, transformers, or other ML libraries. This guarantees
# notebook/CLI parity: identical PYTHONHASHSEED, CUBLAS_WORKSPACE_CONFIG,
# TOKENIZERS_PARALLELISM, torch.use_deterministic_algorithms, and cuDNN flags
# regardless of whether the code runs in JupyterLab or from the CLI.
# See src/repro.py for full rationale.
#
# Failure isolation: run_preflight_checks() is a hard gate that validates ALL
# critical preconditions BEFORE any expensive GPU training begins. This prevents
# wasted GPU hours from misconfigured environments discovered mid-run.
# If preflight fails, the notebook raises immediately with an actionable message.
# --- Step 0: Reproducibility (must be FIRST — before torch import) ---
import os
import sys
from pathlib import Path

# Locate project root by walking up until pyproject.toml is found.
# Idempotent: safe to re-run the cell without drifting further up.
_here = Path.cwd().resolve()
_root = next((p for p in [_here, *_here.parents] if (p / "pyproject.toml").exists()), None)
assert _root is not None, f"pyproject.toml not found at or above {_here}"
os.chdir(_root)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

# Prepend .venv/bin to PATH so subprocess calls to venv CLI tools (dvc, uv,
# pytest, etc.) resolve correctly. Jupyter kernel inherits a minimal PATH
# that excludes the venv's bin dir even though the kernel's Python IS the
# venv's interpreter.
_venv_bin = str(_root / ".venv" / "bin")
if _venv_bin not in os.environ["PATH"]:
    os.environ["PATH"] = f"{_venv_bin}:{os.environ['PATH']}"

# Load .env so preflight sees TARGET_GPU_COUNT, TARGET_VRAM_GB_MIN,
# PYTHONHASHSEED, CUBLAS_WORKSPACE_CONFIG, TOKENIZERS_PARALLELISM.
# override=False → repro.configure() still wins for repro-critical vars.
from dotenv import load_dotenv

load_dotenv(_root / ".env", override=False)

from src.repro import configure as _configure_repro

repro_cfg = _configure_repro(verbose=True)
# --- Step 1: Remaining imports (torch now imported safely after repro config) ---
import logging

from src.environment import (
    REQUIRED_DEPS,
    get_environment_summary,
    run_environment_checks,
    run_preflight_checks,
)
from src.timer import cell_timer

logger = logging.getLogger("cell1")
logger.setLevel(logging.INFO)
if not logger.handlers:
    handler = logging.StreamHandler()
    handler.setFormatter(logging.Formatter("  %(message)s"))
    logger.addHandler(handler)
with cell_timer("Cell 1 — Environment Setup & GPU Verification", logger=logger):
    # --- TDD Contract ---
    logger.info("=" * 60)
    logger.info("  TDD RED→GREEN: Environment Contract")
    logger.info("=" * 60)
    assert run_environment_checks(logger=logger), "Environment contract violated"
    # --- Preflight Gate (hard stop before any expensive training) ---
    # Validates: GPU count/name/VRAM, disk space, repro config integrity,
    # uv.lock hash match, and src/repro.py presence.
    # Raises PreflightError with actionable message on any failure.
    logger.info(f"\n{'=' * 60}")
    logger.info("  Preflight Checks — Failure Isolation Gate")
    logger.info("=" * 60)
    run_preflight_checks(logger=logger, repro_cfg=repro_cfg)
    # --- Repro Config Summary ---
    logger.info(f"\n{'=' * 60}")
    logger.info("  Reproducibility Config (src/repro.configure)")
    logger.info("=" * 60)
    for k, v in repro_cfg.items():
        logger.info(f"  {k:<36} {v}")
    # --- Environment Summary ---
    env = get_environment_summary()
    logger.info(f"\n{'=' * 60}")
    logger.info("  Verified Environment")
    logger.info("=" * 60)
    for pkg, constraint in REQUIRED_DEPS.items():
        logger.info(f"  {pkg:<16} {env[pkg]:<12} (requires {constraint or 'any'})")
    logger.info(f"  {'GPU':<16} {env['gpu']}")
    logger.info(f"  {'GPU Memory':<16} {env['gpu_memory_gb']} GB")
    logger.info(f"  {'CUDA':<16} {env['cuda']}")
    logger.info("\n✓ Environment ready — all preflight checks passed, safe to proceed")

    TDD RED→GREEN: Environment Contract


  [repro] Reproducibility configured:
    PYTHONHASHSEED=0
    CUBLAS_WORKSPACE_CONFIG=:4096:8
    TOKENIZERS_PARALLELISM=false
    deterministic_algorithms=True
    cudnn_benchmark=False
    cudnn_deterministic=True
    random_seed=0
    torch.cuda.manual_seed_all(0) → 1 GPU(s)


  ✓ PASS: Every required dependency must be importable and meet version constraints
  ✓ PASS: CUDA GPU must be detected for training
  ✓ PASS: GPU must have at least 10GB VRAM for transformer fine-tuning
  ✓ PASS: PyTorch must be compiled with CUDA support
  ✓ PASS: Cross-dependency version constraints must be satisfied
  
    Preflight Checks — Failure Isolation Gate
  ✓ PASS: GPU count 1 == TARGET_GPU_COUNT=1 (exact)
  ✓ PASS: GPU[0] NVIDIA L4 | cap (8, 9) | 23.7GB
  ✓ PASS: torch CUDA runtime 11.7
  ✓ PASS: Disk 9223360664.8GB free
  ✓ PASS: src/repro.py importable
  ✓ PASS: repro_cfg['PYTHONHASHSEED'] = '0'
  ✓ PASS: repro_cfg['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
  ✓ PASS: repro_cfg['TOKENIZERS_PARALLELISM'] = 'false'
  ✓ PASS: repro_cfg['deterministic_algorithms'] = True
  ✓ PASS: repro_cfg['cudnn_benchmark'] = False
  ✓ PASS: repro_cfg['cudnn_deterministic'] = True
  ✓ PASS: torch runtime state — deterministic
  ✓ PASS: os.environ['PYTHONHASHSEED'] = '0'
  ✓ PASS: os.environ['C

In [2]:
# Cell: Verify CL bulk data directory on Harvard cluster filesystem
"""
Harvard cluster home directories persist across sessions, so CourtListener
bulk archives live directly at data/raw/cl_bulk/ — no Drive mount or symlink
needed. This cell ensures the directory exists and reports any existing
archives so warm-start runs skip re-download.
"""
import shutil
import time
from pathlib import Path

_t_start = time.perf_counter()

cl_bulk = Path("data/raw/cl_bulk")
cl_bulk.mkdir(parents=True, exist_ok=True)
print(f"cl_bulk dir: {cl_bulk.resolve()}")

total, used, free = shutil.disk_usage(cl_bulk)
print(f"filesystem free: {free / 1e9:.1f} GB / total: {total / 1e9:.1f} GB")

archives = sorted(cl_bulk.glob("*.csv.bz2"))
if archives:
    print(f"existing archives ({len(archives)}):")
    for p in archives:
        print(f"  {p.name}  {p.stat().st_size / 1e9:.2f} GB")
else:
    print("no archives yet — Cell 5 bulk_download will fetch from CourtListener")

print(f"⏱ completed in {time.perf_counter() - _t_start:.1f}s")

cl_bulk dir: /shared/home/phl690/cs1090b_HallucinationLegalRAGChatbots/data/raw/cl_bulk
filesystem free: 9223360664.8 GB / total: 9223372036.9 GB
existing archives (4):
  courts-2025-12-31.csv.bz2  0.00 GB
  dockets-2025-12-31.csv.bz2  4.88 GB
  opinion-clusters-2025-12-31.csv.bz2  2.45 GB
  opinions-2025-12-31.csv.bz2  53.70 GB
⏱ completed in 0.0s


In [3]:
# Cell: CourtListener bulk CSV download — filesystem-persistent, skip if present
"""
Downloads the 4 CourtListener bulk CSV archives into data/raw/cl_bulk/.
Idempotent: skips entirely if all 4 archives present. Pinned to 2025-12-31
snapshot to match the manifest's source_files (avoids triggering cold
re-extraction from a newer snapshot).
"""
import logging
import time

from src.bulk_download import download_bulk_csvs
from src.config import PipelineConfig

_t_start = time.perf_counter()
logging.basicConfig(level=logging.INFO, format="  %(message)s")
log = logging.getLogger("bulk")

cfg = PipelineConfig(
    pinned_courts="bulk-data/courts-2025-12-31.csv.bz2",
    pinned_dockets="bulk-data/dockets-2025-12-31.csv.bz2",
    pinned_clusters="bulk-data/opinion-clusters-2025-12-31.csv.bz2",
    pinned_opinions="bulk-data/opinions-2025-12-31.csv.bz2",
)
cfg.bulk_dir.mkdir(parents=True, exist_ok=True)
log.info("Snapshot: pinned 2025-12-31 (matches manifest)")

need = {"courts-", "dockets-", "opinion-clusters-", "opinions-"}
have = {p.name for p in cfg.bulk_dir.glob("*.csv.bz2")}
matched = {lbl for lbl in need if any(n.startswith(lbl) for n in have)}

if matched == need:
    log.info(f"All 4 bulk CSVs already present in {cfg.bulk_dir} — skipping")
    for p in sorted(cfg.bulk_dir.glob("*.csv.bz2")):
        log.info(f"  {p.name}  {p.stat().st_size / 1e9:.2f} GB")
else:
    log.info(f"Missing: {sorted(need - matched)}")
    latest = cfg.pinned_files
    for label, info in latest.items():
        log.info(f"  {label:<10} {info['key']}")
    paths = download_bulk_csvs(latest, config=cfg, logger=log)
    for label, p in paths.items():
        log.info(f"  {label:<10} -> {p}")

print(f"⏱ completed in {time.perf_counter() - _t_start:.1f}s")

  Snapshot: pinned 2025-12-31 (matches manifest)
  All 4 bulk CSVs already present in data/raw/cl_bulk — skipping
    courts-2025-12-31.csv.bz2  0.00 GB
    dockets-2025-12-31.csv.bz2  4.88 GB
    opinion-clusters-2025-12-31.csv.bz2  2.45 GB
    opinions-2025-12-31.csv.bz2  53.70 GB


⏱ completed in 0.0s


In [4]:
# Cell: Filter chain + extraction + manifest (Harvard filesystem-persistent shards)
"""
End-to-end Stage 1-2 data pipeline: pinned snapshot → filter chain →
extract → manifest → TDD contract tests. Produces NDJSON shards in
data/raw/cl_federal_appellate_bulk/.

Fast-path: if manifest.json + shards present and source_files match the
pinned 2025-12-31 snapshot, run_pipeline returns immediately.
"""
import logging
import sys

from src.config import PipelineConfig
from src.data_contracts import run_all_contracts
from src.dataset_card import write_dataset_card
from src.dvc_tracking import DVCTrackingError, track_shard_directory
from src.pipeline import run_pipeline, validate_pipeline
from src.timer import cell_timer


def _get_cell_logger():
    lg = logging.getLogger("cell_pipeline")
    lg.setLevel(logging.INFO)
    if not lg.handlers:
        h = logging.StreamHandler(sys.stdout)
        h.setFormatter(logging.Formatter("  %(message)s"))
        lg.addHandler(h)
    lg.propagate = False
    return lg


logger = _get_cell_logger()
cfg = PipelineConfig(
    pinned_courts="bulk-data/courts-2025-12-31.csv.bz2",
    pinned_dockets="bulk-data/dockets-2025-12-31.csv.bz2",
    pinned_clusters="bulk-data/opinion-clusters-2025-12-31.csv.bz2",
    pinned_opinions="bulk-data/opinions-2025-12-31.csv.bz2",
)

with cell_timer("Pipeline (filter + extract + manifest)", logger=logger):
    logger.info("=" * 60)
    logger.info("  run_pipeline (pinned 2025-12-31; fast-path if manifest + shards valid)")
    logger.info("=" * 60)
    manifest = run_pipeline(config=cfg, logger=logger)

    logger.info("\n" + "=" * 60)
    logger.info("  validate_pipeline (TDD contract tests)")
    logger.info("=" * 60)
    validate_pipeline(config=cfg, manifest_data=manifest, logger=logger, shard_strategy="sample")
    logger.info("OK contract tests passed")

    logger.info("\n" + "=" * 60)
    logger.info("  DVC tracking (src.dvc_tracking)")
    logger.info("=" * 60)
    try:
        track_shard_directory(cfg.shard_dir, repo_root=".", push=False)
        logger.info("  OK shard_dir versioned (or already tracked)")
    except DVCTrackingError as e:
        logger.info(f"  SKIP dvc: {e}")

    logger.info("\n" + "=" * 60)
    logger.info("  HF dataset card (src.dataset_card)")
    logger.info("=" * 60)
    card_path = write_dataset_card(manifest, cfg.shard_dir)
    logger.info(f"  OK wrote {card_path}")

    logger.info("\n" + "=" * 60)
    logger.info("  Statistical data contracts (src.data_contracts)")
    logger.info("=" * 60)
    for r in run_all_contracts(manifest, strict=False):
        status = "PASS" if r.passed else "FAIL"
        logger.info(f"  {r.name:<28} {status} - {r.message}")

    logger.info("\n" + "=" * 60)
    logger.info("  Summary")
    logger.info("=" * 60)
    logger.info(f"  snapshot: {manifest['source_files']['opinions']}")
    logger.info(f"  git_rev:  {manifest['run_metadata']['git_revision'][:12]}")
    logger.info(f"  shards:   {manifest['num_shards']}")
    logger.info(f"  cases:    {manifest['num_cases']:,}")
    logger.info(f"  scanned:  {manifest.get('scanned', 0):,}")
    logger.info(f"  circuits: {len(manifest.get('court_distribution', {}))}")
    tls = manifest.get("text_length_stats", {})
    if tls:
        logger.info(
            f"  text len: mean={tls.get('mean', 0):,} median={tls.get('median', 0):,} p95={tls.get('p95', 0):,}"
        )

    run_pipeline (pinned 2025-12-31; fast-path if manifest + shards valid)
  ✓ Already complete: 1,465,484 cases, 159 shards verified
  
    validate_pipeline (TDD contract tests)
  ✓ PASS: Shard directory must exist
  ✓ PASS: Manifest must exist
  ✓ PASS: At least one shard present
  ✓ PASS: Sufficient opinions
  ✓ PASS: Valid JSON
  ✓ PASS: Text present
  ✓ PASS: Text substantive
  ✓ PASS: Provenance metadata
  ✓ PASS: Raw + normalized text + flags
  ✓ PASS: Text source tracked
  ✓ PASS: Multiple circuits
  ✓ PASS: Schema consistent
  ✓ PASS: Checksums match
  OK contract tests passed
  
    DVC tracking (src.dvc_tracking)
    SKIP dvc: dvc add failed (exit 1): ERROR: bad DVC file name 'data/raw/cl_federal_appellate_bulk.dvc' is git-ignored.
  
    HF dataset card (src.dataset_card)
    OK wrote data/raw/cl_federal_appellate_bulk/README.md
  
    Statistical data contracts (src.data_contracts)
    row_count_floor              PASS - 1,465,484 >= 10,000
    court_balance              

In [5]:
import json
from pathlib import Path

m = json.load(open("data/raw/cl_federal_appellate_bulk/manifest.json"))
print("disk manifest keys:", sorted(m.keys())[:10], "...")
print("court_distribution type:", type(m.get("court_distribution")), "len:", len(m.get("court_distribution", {})))
print("text_length_stats:", m.get("text_length_stats"))

disk manifest keys: ['checksum', 'court_distribution', 'federal_courts', 'filter_chain', 'num_cases', 'num_shards', 'ocr_extracted_count', 'opinion_type_distribution', 'precedential_status_distribution', 'run_metadata'] ...
court_distribution type: <class 'dict'> len: 0
text_length_stats: {}
